In [22]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the dataset
dataset_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single'

# Check if the path exists and list the first few files
if os.path.exists(dataset_path):
    print(f"Successfully accessed: {dataset_path}")
    files = os.listdir(dataset_path)
    print(f"Total items found: {len(files)}")
    print("Sample files:", files[:5])
else:
    print(f"Path not found: {dataset_path}. Please ensure the folder name is correct.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully accessed: /content/drive/MyDrive/Colab Notebooks/MVSA_Single
Total items found: 17
Sample files: ['data', 'labelResultAll.txt', 'mvsa_single_processed.parquet', 'checkpoints', 'vilbert_final']


In [24]:
# =====================================================
# Cell - Check Parquet Dataset
# =====================================================
import pandas as pd
import os
import sys
import warnings

try:
    parquet_path = CONFIG.get('parquet_path', '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed_224.parquet')
except NameError:
    parquet_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed_224.parquet'

if not os.path.exists(parquet_path):
    msg = f"Parquet file {parquet_path} not found. Please run 'mvsa-to-parquet.ipynb' first to generate the dataset."
    warnings.warn(msg, UserWarning)
    sys.exit(f"FileNotFoundError: {msg}")
else:
    print(f"File parquet found, loading from: {parquet_path}")
    # Optional preview load
    df = pd.read_parquet(parquet_path)
    print(f"Total entries loaded: {len(df)}")
    if 'final_sentiment' in df.columns:
        print("\nSentiment Distribution:")
        print(df['final_sentiment'].value_counts())
    display(df.head())


File parquet sudah ada, memuat dari: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Total data dimuat: 4511

Distribusi sentimen:
final_sentiment
positive    2683
negative    1358
neutral      470
Name: count, dtype: int64


,ID,text_sentiment,image_sentiment,final_sentiment,image_bytes,text_content
0,1,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,How I feel today #legday #jelly #aching #gym
1,2,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,grattis min griskulting!!!???? va bara tvungen...
2,3,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,RT @polynminion: The moment I found my favouri...
3,4,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,#escort We have a young and energetic team and...
4,5,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,"RT @chrisashaffer: Went to SSC today to be a ""..."


In [25]:
# Cell 4 - Install & Import Dependencies
%pip install torch torchvision scikit-learn pyarrow tqdm transformers -q

import os, io, random, warnings, json
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision.models as tv_models
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight as _cwf

from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print("All libraries imported successfully.")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

All libraries imported successfully.
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU Memory      : 15.6 GB


In [26]:
# Cell 5 - MSA mode configuration and seed setup
CONFIG = {
    # Paths
    "parquet_path": parquet_path,
    "checkpoint_dir": "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/bert_lstm_checkpoints",
    "artifact_dir": "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/bert_lstm_final",

    # Mode identity
    "mode_name": "msa_bert_resnet50_lstm",

    # Text encoder (BERT)
    "bert_model": "bert-base-uncased",
    "bert_hidden_dim": 768,
    "text_proj_dim": 256,
    "bert_finetune": True,
    "text_pooling": "cls_token",
    "max_seq_len": 64,

    # Image encoder (ResNet50 + visual LSTM)
    "image_backbone": "resnet50",
    "image_feature_dim": 2048,
    "image_proj_dim": 256,
    "image_input_size": 224,
    "resnet_finetune": True,
    "visual_lstm_hidden_dim": 256,
    "visual_lstm_layers": 2,
    "visual_lstm_bidirectional": False,

    # Co-attention
    "d_model": 256,
    "co_attn_type": "multi-head",
    "num_heads": 8,
    "co_attn_layers": 3,
    "co_attn_ffn_dim": 1024,
    "co_attn_dropout": 0.10,

    # Fusion and classifier
    "fusion_strategy": "concatenation",
    "fusion_input_dim": 256 * 2,
    "classifier_hidden": 256,
    "num_classes": 3,
    "classifier_dropout": 0.25,

    # Training
    "epochs": 30,
    "batch_size": 16,
    "lr_main": 1e-4,
    "lr_bert_finetune": 2e-5,
    "lr_resnet_finetune": 1e-5,
    "weight_decay": 0.01,
    "gradient_clip": 1.0,
    "early_stop_patience": 7,
    "scheduler": "CosineAnnealingLR",

    # Data split
    "train_ratio": 0.80,
    "val_ratio": 0.10,
    "test_ratio": 0.10,

    # Reproducibility
    "seed": 42,
    "num_workers": 2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

_vout = CONFIG["visual_lstm_hidden_dim"] * (2 if CONFIG["visual_lstm_bidirectional"] else 1)
if CONFIG["text_proj_dim"] != CONFIG["d_model"]:
    raise ValueError("text_proj_dim must equal d_model for co-attention.")
if _vout != CONFIG["d_model"]:
    raise ValueError("visual_lstm_hidden_dim * directions must equal d_model for co-attention.")
if CONFIG["fusion_input_dim"] != CONFIG["d_model"] * 2:
    raise ValueError("fusion_input_dim must equal d_model * 2.")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["artifact_dir"], exist_ok=True)

SEP = "=" * 62
print(SEP)
print("   Multimodal BERT + ResNet50-LSTM Sentiment - Configuration")
print(SEP)
print(f"  Mode                    : {CONFIG['mode_name']}")
print(f"  Device                  : {CONFIG['device'].upper()}")
if torch.cuda.is_available():
    print(f"  GPU                     : {torch.cuda.get_device_name(0)}")

_vdir = "Bidirectional" if CONFIG["visual_lstm_bidirectional"] else "Unidirectional"

print("\n  -- Text Encoder (BERT) --------------------------------")
print(f"  Backbone                : {CONFIG['bert_model']} (HuggingFace pretrained)")
print(f"  BERT hidden dim         : {CONFIG['bert_hidden_dim']}")
print(f"  Projected dim           : {CONFIG['text_proj_dim']}  (shared d_model)")
print(f"  BERT fine-tune          : {CONFIG['bert_finetune']}")
print(f"  Pooling strategy        : {CONFIG['text_pooling']}")
print(f"  Max sequence length     : {CONFIG['max_seq_len']}")

print("\n  -- Image Encoder (ResNet50 + LSTM) --------------------")
print(f"  Backbone                : {CONFIG['image_backbone']} (pretrained ImageNet)")
print(f"  Feature dim (raw)       : {CONFIG['image_feature_dim']}")
print(f"  Projected dim           : {CONFIG['image_proj_dim']}")
print(f"  Input image size        : {CONFIG['image_input_size']}x{CONFIG['image_input_size']}")
print(f"  ResNet fine-tune        : {CONFIG['resnet_finetune']}")
print(f"  Visual LSTM direction   : {_vdir}")
print(f"  Visual LSTM hidden dim  : {CONFIG['visual_lstm_hidden_dim']}  -> output {_vout}")
print(f"  Visual LSTM layers      : {CONFIG['visual_lstm_layers']}")

print("\n  -- Co-Attention Mechanism ------------------------------")
print(f"  Type                    : {CONFIG['co_attn_type']} (bidirectional cross-modal)")
print(f"  Number of heads         : {CONFIG['num_heads']}")
print(f"  Number of layers        : {CONFIG['co_attn_layers']}")
print(f"  d_model                 : {CONFIG['d_model']}  (shared text/image dimension)")
print(f"  Head dimension          : {CONFIG['d_model'] // CONFIG['num_heads']}  (d_model / num_heads)")
print(f"  FFN hidden dim          : {CONFIG['co_attn_ffn_dim']}")
print(f"  Dropout                 : {CONFIG['co_attn_dropout']}")

print("\n  -- Fusion and Classifier -------------------------------")
print(f"  Fusion strategy         : {CONFIG['fusion_strategy']}")
print(f"  Fusion input dim        : {CONFIG['fusion_input_dim']}")
print(f"  Classifier hidden       : {CONFIG['classifier_hidden']}")
print(f"  Num output classes      : {CONFIG['num_classes']}  (neg / neu / pos)")
print(f"  Classifier dropout      : {CONFIG['classifier_dropout']}")

print("\n  -- Training Hyperparameters ----------------------------")
print(f"  Epochs                  : {CONFIG['epochs']}")
print(f"  Batch size              : {CONFIG['batch_size']}")
print(f"  LR (main)               : {CONFIG['lr_main']}")
print(f"  LR (BERT fine-tune)     : {CONFIG['lr_bert_finetune']}")
print(f"  LR (ResNet fine-tune)   : {CONFIG['lr_resnet_finetune']}")
print(f"  Weight decay            : {CONFIG['weight_decay']}")
print(f"  Gradient clip           : {CONFIG['gradient_clip']}")
print(f"  LR scheduler            : {CONFIG['scheduler']}")
print(f"  Early-stop patience     : {CONFIG['early_stop_patience']} epochs")

print("\n  -- Data ------------------------------------------------")
print(f"  Split (train/val/test)  : {CONFIG['train_ratio']}/{CONFIG['val_ratio']}/{CONFIG['test_ratio']}")
print(f"  Random seed             : {CONFIG['seed']}")
print(SEP)

   Multimodal BERT + ResNet50-LSTM Sentiment - Configuration
  Mode                    : msa_bert_resnet50_lstm
  Device                  : CUDA
  GPU                     : Tesla T4

  -- Text Encoder (BERT) --------------------------------
  Backbone                : bert-base-uncased (HuggingFace pretrained)
  BERT hidden dim         : 768
  Projected dim           : 256  (shared d_model)
  BERT fine-tune          : True
  Pooling strategy        : cls_token
  Max sequence length     : 64

  -- Image Encoder (ResNet50 + LSTM) --------------------
  Backbone                : resnet50 (pretrained ImageNet)
  Feature dim (raw)       : 2048
  Projected dim           : 256
  Input image size        : 224x224
  ResNet fine-tune        : True
  Visual LSTM direction   : Unidirectional
  Visual LSTM hidden dim  : 256  -> output 256
  Visual LSTM layers      : 2

  -- Co-Attention Mechanism ------------------------------
  Type                    : multi-head (bidirectional cross-modal)
  Num

In [27]:
# Cell 6 - Load Parquet, Validate Schema, Label Encoding, Split
print(f"Loading parquet from: {CONFIG['parquet_path']}")
df_full = pd.read_parquet(CONFIG["parquet_path"])
print(f"Total rows loaded : {len(df_full)}")
print(f"Columns           : {list(df_full.columns)}")

required_cols = ["image_bytes", "text_content", "final_sentiment"]
missing_cols = [c for c in required_cols if c not in df_full.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in parquet: {missing_cols}")

before = len(df_full)
df_full = df_full.dropna(subset=required_cols).reset_index(drop=True)
print(f"Rows after dropna : {len(df_full)}  (dropped {before - len(df_full)})")

LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_IMAP = {v: k for k, v in LABEL_MAP.items()}

df_full["label"] = df_full["final_sentiment"].astype(str).str.strip().str.lower().map(LABEL_MAP)
if df_full["label"].isna().any():
    bad_rows = int(df_full["label"].isna().sum())
    print(f"WARN: {bad_rows} rows have unknown sentiment label. Dropping them.")
    df_full = df_full.dropna(subset=["label"]).reset_index(drop=True)
df_full["label"] = df_full["label"].astype(int)

print("\nLabel distribution (full data):")
for name, idx in LABEL_MAP.items():
    cnt = int((df_full["label"] == idx).sum())
    pct = (cnt / len(df_full) * 100) if len(df_full) > 0 else 0.0
    print(f"  {idx}  {name:<10}  {cnt:>5}  ({pct:.1f}%)")

df_train, df_tmp = train_test_split(
    df_full,
    test_size=(CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_full["label"],
    random_state=CONFIG["seed"],
)
df_val, df_test = train_test_split(
    df_tmp,
    test_size=CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_tmp["label"],
    random_state=CONFIG["seed"],
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print("\nData split (stratified):")
print(f"  Train : {len(df_train):>5} rows")
print(f"  Val   : {len(df_val):>5} rows")
print(f"  Test  : {len(df_test):>5} rows")

def _show_split_dist(name, split_df):
    print(f"\n{name} label distribution:")
    for cls_name, cls_idx in LABEL_MAP.items():
        cnt = int((split_df["label"] == cls_idx).sum())
        pct = (cnt / len(split_df) * 100) if len(split_df) > 0 else 0.0
        print(f"  {cls_idx}  {cls_name:<10}  {cnt:>5}  ({pct:.1f}%)")

_show_split_dist("Train", df_train)
_show_split_dist("Val", df_val)
_show_split_dist("Test", df_test)

Loading parquet from: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Total rows loaded : 4511
Columns           : ['ID', 'text_sentiment', 'image_sentiment', 'final_sentiment', 'image_bytes', 'text_content']
Rows after dropna : 4511  (dropped 0)

Label distribution (full data):
  0  negative     1358  (30.1%)
  1  neutral       470  (10.4%)
  2  positive     2683  (59.5%)

Data split (stratified):
  Train :  3608 rows
  Val   :   451 rows
  Test  :   452 rows

Train label distribution:
  0  negative     1086  (30.1%)
  1  neutral       376  (10.4%)
  2  positive     2146  (59.5%)

Val label distribution:
  0  negative      136  (30.2%)
  1  neutral        47  (10.4%)
  2  positive      268  (59.4%)

Test label distribution:
  0  negative      136  (30.1%)
  1  neutral        47  (10.4%)
  2  positive      269  (59.5%)


In [28]:
# Cell 7 - Tokenizer, Dataset, and DataLoaders
tokenizer = AutoTokenizer.from_pretrained(CONFIG["bert_model"], use_fast=True)

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_input_size"], CONFIG["image_input_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_input_size"], CONFIG["image_input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])

class MVSABertImageDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, image_transform):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        raw = row["image_bytes"]
        raw = raw if isinstance(raw, bytes) else bytes(raw)
        image = Image.open(io.BytesIO(raw)).convert("RGB")
        image_tensor = self.transform(image)

        text = str(row["text_content"])
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        token_type_ids = encoded.get("token_type_ids", torch.zeros_like(input_ids)).squeeze(0)

        return {
            "image": image_tensor,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
        }

train_dataset = MVSABertImageDataset(
    df_train, tokenizer=tokenizer, max_len=CONFIG["max_seq_len"], image_transform=train_transform
)
val_dataset = MVSABertImageDataset(
    df_val, tokenizer=tokenizer, max_len=CONFIG["max_seq_len"], image_transform=eval_transform
)
test_dataset = MVSABertImageDataset(
    df_test, tokenizer=tokenizer, max_len=CONFIG["max_seq_len"], image_transform=eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

print(f"Train batches : {len(train_loader):>4}  ({len(train_dataset)} samples)")
print(f"Val   batches : {len(val_loader):>4}  ({len(val_dataset)} samples)")
print(f"Test  batches : {len(test_loader):>4}  ({len(test_dataset)} samples)")

sample_batch = next(iter(train_loader))
print("\nBatch shapes:")
print(f"  image         : {sample_batch['image'].shape}")
print(f"  input_ids     : {sample_batch['input_ids'].shape}")
print(f"  attention_mask: {sample_batch['attention_mask'].shape}")
print(f"  token_type_ids: {sample_batch['token_type_ids'].shape}")
print(f"  label         : {sample_batch['label'].shape}")
del sample_batch

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches :  226  (3608 samples)
Val   batches :   29  (451 samples)
Test  batches :   29  (452 samples)

Batch shapes:
  image         : torch.Size([16, 3, 224, 224])
  input_ids     : torch.Size([16, 64])
  attention_mask: torch.Size([16, 64])
  token_type_ids: torch.Size([16, 64])
  label         : torch.Size([16])


In [29]:
# Cell 8 - Model Architecture (BERT Text + Visual LSTM + Co-Attention)
class BERTTextEncoder(nn.Module):
    def __init__(self, model_name, proj_dim, finetune=True):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        if not finetune:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.proj = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, proj_dim),
            nn.LayerNorm(proj_dim),
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        return self.proj(outputs.last_hidden_state)

class ImageEncoderLSTM(nn.Module):
    def __init__(self, proj_dim, hidden_dim, num_layers, finetune=True, bidirectional=False):
        super().__init__()
        resnet = tv_models.resnet50(weights=tv_models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        if not finetune:
            for p in self.backbone.parameters():
                p.requires_grad = False

        for m in self.backbone.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()

        self.proj = nn.Sequential(
            nn.Linear(2048, proj_dim),
            nn.LayerNorm(proj_dim),
        )
        self.lstm = nn.LSTM(
            input_size=proj_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=0.3 if num_layers > 1 else 0.0,
        )

    def forward(self, pixel_values):
        feat = self.backbone(pixel_values)
        flat = feat.flatten(2).permute(0, 2, 1)
        proj = self.proj(flat)
        out, _ = self.lstm(proj)
        return out

    def train(self, mode=True):
        super().train(mode)
        for m in self.backbone.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

class CoAttentionLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads})")

        self.t2i_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.t2i_norm1 = nn.LayerNorm(d_model)
        self.t2i_ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
        self.t2i_norm2 = nn.LayerNorm(d_model)

        self.i2t_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.i2t_norm1 = nn.LayerNorm(d_model)
        self.i2t_ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
        self.i2t_norm2 = nn.LayerNorm(d_model)

    def forward(self, text_feats, image_feats, text_pad_mask=None):
        if text_pad_mask is not None:
            all_masked = text_pad_mask.all(dim=1)
            if all_masked.any():
                text_pad_mask = text_pad_mask.clone()
                text_pad_mask[all_masked] = False

        t_out, _ = self.t2i_attn(text_feats, image_feats, image_feats)
        t_out = self.t2i_norm1(text_feats + t_out)
        t_out = self.t2i_norm2(t_out + self.t2i_ffn(t_out))

        i_out, _ = self.i2t_attn(
            image_feats, text_feats, text_feats, key_padding_mask=text_pad_mask
        )
        i_out = self.i2t_norm1(image_feats + i_out)
        i_out = self.i2t_norm2(i_out + self.i2t_ffn(i_out))

        return t_out, i_out

class MultimodalSentimentModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.text_pooling = config.get("text_pooling", "cls_token")
        self.text_encoder = BERTTextEncoder(
            model_name=config["bert_model"],
            proj_dim=config["text_proj_dim"],
            finetune=config["bert_finetune"],
        )
        self.image_encoder = ImageEncoderLSTM(
            proj_dim=config["image_proj_dim"],
            hidden_dim=config["visual_lstm_hidden_dim"],
            num_layers=config["visual_lstm_layers"],
            finetune=config["resnet_finetune"],
            bidirectional=config["visual_lstm_bidirectional"],
        )
        self.co_attn_layers = nn.ModuleList([
            CoAttentionLayer(
                d_model=config["d_model"],
                num_heads=config["num_heads"],
                ffn_dim=config["co_attn_ffn_dim"],
                dropout=config["co_attn_dropout"],
            )
            for _ in range(config["co_attn_layers"])
        ])

        self.classifier = nn.Sequential(
            nn.LayerNorm(config["fusion_input_dim"]),
            nn.Linear(config["fusion_input_dim"], config["classifier_hidden"]),
            nn.ReLU(),
            nn.Dropout(config["classifier_dropout"]),
            nn.Linear(config["classifier_hidden"], config["num_classes"]),
        )

    def forward(self, input_ids, attention_mask, images, token_type_ids=None):
        text_feats = self.text_encoder(input_ids, attention_mask, token_type_ids)
        image_feats = self.image_encoder(images)

        text_pad_mask = (attention_mask == 0) if attention_mask is not None else None
        for layer in self.co_attn_layers:
            text_feats, image_feats = layer(text_feats, image_feats, text_pad_mask=text_pad_mask)

        if self.text_pooling == "cls_token":
            text_pool = text_feats[:, 0, :]
        elif text_pad_mask is not None:
            valid_mask = (~text_pad_mask).float().unsqueeze(-1)
            text_pool = (text_feats * valid_mask).sum(1) / valid_mask.sum(1).clamp(min=1.0)
        else:
            text_pool = text_feats.mean(1)

        image_pool = image_feats.mean(1)
        fused = torch.cat([text_pool, image_pool], dim=-1)
        return self.classifier(fused)

    def train(self, mode=True):
        super().train(mode)
        self.image_encoder.train(mode)
        return self

device = torch.device(CONFIG["device"])
model = MultimodalSentimentModel(CONFIG).to(device)

total_p = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_p:,}")
print(f"Trainable parameters: {trainable_p:,}")

model.eval()
with torch.no_grad():
    b = next(iter(train_loader))
    ids = b["input_ids"].to(device)
    am = b["attention_mask"].to(device)
    tt = b["token_type_ids"].to(device)
    imgs = b["image"].to(device)
    logits = model(ids, am, imgs, token_type_ids=tt)
print(f"Forward pass OK | logits shape: {tuple(logits.shape)} (expect [B, 3])")
del b, ids, am, tt, imgs, logits
model.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 35.1MB/s]


Total parameters    : 138,057,539
Trainable parameters: 138,057,539
Forward pass OK | logits shape: (16, 3) (expect [B, 3])


MultimodalSentimentModel(
  (text_encoder): BERTTextEncoder(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=7

In [30]:
# Cell 9 - Training Loop
resnet_param_ids = set(id(p) for p in model.image_encoder.backbone.parameters())
bert_param_ids = set(id(p) for p in model.text_encoder.bert.parameters())

resnet_params = [p for p in model.parameters() if id(p) in resnet_param_ids and p.requires_grad]
bert_params = [p for p in model.parameters() if id(p) in bert_param_ids and p.requires_grad]
main_params = [
    p for p in model.parameters()
    if id(p) not in resnet_param_ids and id(p) not in bert_param_ids and p.requires_grad
]

optimizer = AdamW([
    {"params": main_params, "lr": CONFIG["lr_main"]},
    {"params": bert_params, "lr": CONFIG["lr_bert_finetune"]},
    {"params": resnet_params, "lr": CONFIG["lr_resnet_finetune"]},
], weight_decay=CONFIG["weight_decay"])

scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)

cw = _cwf(
    class_weight="balanced",
    classes=np.arange(CONFIG["num_classes"]),
    y=df_train["label"].values,
)
class_weights = torch.tensor(cw, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(f"Class weights - negative={cw[0]:.3f}  neutral={cw[1]:.3f}  positive={cw[2]:.3f}")

def evaluate(loader, model, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            am = batch["attention_mask"].to(device)
            tt = batch["token_type_ids"].to(device)
            imgs = batch["image"].to(device)
            labels = batch["label"].to(device)

            logits = model(ids, am, imgs, token_type_ids=tt)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / max(len(loader), 1)
    acc = accuracy_score(all_labels, all_preds) if all_labels else 0.0
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0) if all_labels else 0.0
    return avg_loss, acc, f1

best_val_f1 = 0.0
patience_counter = 0
best_ckpt_path = os.path.join(CONFIG["checkpoint_dir"], "bert_lstm_best.pt")
history = []

header = (
    f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  "
    f"{'Val Loss':>8}  {'Val Acc':>7}  {'Val F1':>7}  {'LR(main)':>9}  {'Status'}"
)
print(header)
print("-" * len(header))

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:>2}/{CONFIG['epochs']}", leave=False)
    for batch in pbar:
        ids = batch["input_ids"].to(device)
        am = batch["attention_mask"].to(device)
        tt = batch["token_type_ids"].to(device)
        imgs = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(ids, am, imgs, token_type_ids=tt)
        loss = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
        optimizer.step()

        train_loss += loss.item()
        train_preds.extend(logits.detach().argmax(dim=-1).cpu().tolist())
        train_labels.extend(labels.cpu().tolist())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    tr_loss = train_loss / max(len(train_loader), 1)
    tr_acc = accuracy_score(train_labels, train_preds) if train_labels else 0.0
    val_loss, val_acc, val_f1 = evaluate(val_loader, model, criterion, device)
    cur_lr = optimizer.param_groups[0]["lr"]

    status = ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_f1": val_f1,
            "val_acc": val_acc,
            "config": CONFIG,
            "label_map": LABEL_MAP,
            "label_imap": LABEL_IMAP,
            "tokenizer_name": CONFIG["bert_model"],
        }, best_ckpt_path)
        status = "best saved"
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stop_patience"]:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {CONFIG['early_stop_patience']} epochs)")
            break

    history.append({
        "epoch": epoch,
        "train_loss": tr_loss,
        "train_acc": tr_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
    })

    print(
        f"{epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  "
        f"{val_loss:>8.4f}  {val_acc:>7.4f}  {val_f1:>7.4f}  {cur_lr:>9.2e}  {status}"
    )

print(f"\nTraining complete. Best val macro-F1: {best_val_f1:.4f}")
print(f"Best checkpoint path: {best_ckpt_path}")

Class weights - negative=1.107  neutral=3.199  positive=0.560
Epoch  Train Loss  Train Acc  Val Loss  Val Acc   Val F1   LR(main)  Status
---------------------------------------------------------------------------


Epoch  1/30:   0%|          | 0/226 [00:00<?, ?it/s]

    1      0.9374     0.6073    0.8937   0.6519   0.5706   9.97e-05  best saved


Epoch  2/30:   0%|          | 0/226 [00:00<?, ?it/s]

    2      0.6327     0.7694    0.8820   0.7184   0.6393   9.89e-05  best saved


...

    6      0.1618     0.9656    1.9916   0.7007   0.6106   9.05e-05  


Epoch  7/30:   0%|          | 0/226 [00:00<?, ?it/s]

    7      0.1337     0.9759    2.2080   0.7007   0.6004   8.73e-05  


Epoch  8/30:   0%|          | 0/226 [00:00<?, ?it/s]


Early stopping at epoch 8 (no improvement for 5 epochs)

Training complete. Best val macro-F1: 0.6403
Best checkpoint path: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/bert_lstm_checkpoints/bert_lstm_best.pt


In [31]:
# Cell 10 - Evaluation on Test Set and Save Artifacts
print(f"Loading best checkpoint: {best_ckpt_path}")
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
print(
    f"Checkpoint epoch: {ckpt['epoch']}  val_f1={ckpt['val_f1']:.4f}  val_acc={ckpt['val_acc']:.4f}"
)

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        ids = batch["input_ids"].to(device)
        am = batch["attention_mask"].to(device)
        tt = batch["token_type_ids"].to(device)
        imgs = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = model(ids, am, imgs, token_type_ids=tt)
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

class_names = [LABEL_IMAP[i] for i in range(CONFIG["num_classes"])]
test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

SEP = "=" * 62
print(f"\n{SEP}")
print("  Test Set Results - Multimodal BERT + ResNet50-LSTM")
print(SEP)
print(f"  Accuracy  (overall)    : {test_acc:.4f}  ({test_acc * 100:.2f}%)")
print(f"  Macro F1  (unweighted) : {test_f1:.4f}")
print("\n  Per-Class Classification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(all_labels, all_preds)
print("  Confusion Matrix (rows=true, cols=pred):")
col_w = max(len(n) for n in class_names) + 2
header_row = " " * 12 + "  ".join(f"{n:>{col_w}}" for n in class_names)
print(header_row)
for i, row_vals in enumerate(cm):
    row_str = "  ".join(f"{v:{col_w}d}" for v in row_vals)
    print(f"  {class_names[i]:>10}  {row_str}")

if history:
    hist_df = pd.DataFrame(history)
    print("\n  Training History (last 5 epochs):")
    print(hist_df.tail(5).to_string(index=False))

print(f"\n{SEP}")

SAVE_DIR = CONFIG["artifact_dir"]
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": CONFIG,
    "label_map": LABEL_MAP,
    "label_imap": LABEL_IMAP,
    "best_val_f1": best_val_f1,
    "test_acc": test_acc,
    "test_f1": test_f1,
    "history": history,
}, os.path.join(SAVE_DIR, "bert_lstm_full.pt"))

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "bert_lstm_weights_only.pt"))
tokenizer.save_pretrained(SAVE_DIR)

cfg_json = {k: v for k, v in CONFIG.items() if not callable(v)}
cfg_json["label_map"] = LABEL_MAP
cfg_json["label_imap"] = {str(k): v for k, v in LABEL_IMAP.items()}
with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump(cfg_json, f, indent=2)

for fname in ["bert_lstm_full.pt", "bert_lstm_weights_only.pt", "config.json"]:
    fpath = os.path.join(SAVE_DIR, fname)
    mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname:<35}  {mb:.1f} MB")

print(f"\nArtifacts saved to: {SAVE_DIR}")
print(SEP)

Loading best checkpoint: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/bert_lstm_checkpoints/bert_lstm_best.pt
Checkpoint epoch: 3  val_f1=0.6403  val_acc=0.6940


Testing:   0%|          | 0/29 [00:00<?, ?it/s]


  Test Set Results - Multimodal BERT + ResNet50-LSTM
  Accuracy  (overall)    : 0.7522  (75.22%)
  Macro F1  (unweighted) : 0.6733

  Per-Class Classification Report:
              precision    recall  f1-score   support

    negative     0.6871    0.7426    0.7138       136
     neutral     0.5116    0.4681    0.4889        47
    positive     0.8282    0.8067    0.8173       269

    accuracy                         0.7522       452
   macro avg     0.6756    0.6725    0.6733       452
weighted avg     0.7528    0.7522    0.7520       452

  Confusion Matrix (rows=true, cols=pred):
              negative     neutral    positive
    negative         101           6          29
     neutral           9          22          16
    positive          37          15         217

  Training History (last 5 epochs):
 epoch  train_loss  train_acc  val_loss  val_acc   val_f1
     3    0.367052   0.881652  1.043791 0.694013 0.640254
     4    0.232679   0.935144  1.530369 0.687361 0.601327
   